In [1]:
#imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Question 1
*(25)*

## 1.1) Objective:
Mutuka Automotive manually evaluates a large volume of second-hand vehicles to estimate their price. This wastes man-hours, is not scalable, and can be automated.
First, we import the catalogue and check if the evaluation factors are present; an integrity check will be used for this. Then make a list of uncalculable vehicles and calculable vehicles by creating two new pandas arrays. We calculate the calculability by comparing the length of the arrays. We run the cost estimation function and compare that to manually estimated prices to find the error margin.
A *successful solution* can parse the catalogue and estimate the value of vehicles or refer to manual evaluation if an anomaly is detected. More than 80% of vehicles should be able to be evaluated; calculability. Evaluation should be within a 10% error margin. *(5)*

## 1.2) Integrity Assesment:

In [2]:
df = pd.read_csv('car_pricing_datasets.csv')  #import dataset

# missing values
missing_count = df.isnull().sum()  # counts all missing per column
missing_count_pct = (missing_count / len(df)) * 100  # calculates missing percentage per column

print('\n', '-' * 50, '\n', ' ' * 15, 'Missing values:\n', '-' * 50)
print(missing_count)
print('\n', '-' * 50, '\n', ' ' * 12, 'Missing Percentage:\n', '-' * 50)
print(missing_count_pct)

missing_df = df[df.isnull().any(axis=1)].copy()  # makes data frame with all items with missing values
missing_pct = (len(missing_df) / len(df)) * 100  # calculates missing percentage total
df_12 = df.dropna().copy()  # dataframe with missing values removed
# new dataframe for every question so I can reference old ones

print('\n', '-' * 50, '\n',
      '  Missing Row Percentage:',missing_pct,'%\n', '-' * 50)



 -------------------------------------------------- 
                 Missing values:
 --------------------------------------------------
car_ID               9
symboling            6
CarName              7
fueltype            10
aspiration           2
doornumber           7
carbody             16
drivewheel           4
enginelocation       4
wheelbase            5
carlength           12
carwidth             7
carheight            3
curbweight           7
enginetype           2
cylindernumber      10
enginesize           9
fuelsystem           6
boreratio            7
stroke               8
compressionratio     6
horsepower           5
peakrpm              6
citympg              8
highwaympg          14
price                0
dtype: int64

 -------------------------------------------------- 
              Missing Percentage:
 --------------------------------------------------
car_ID              4.390244
symboling           2.926829
CarName             3.414634
fueltype            4.8

In [3]:
# duplicate records
duplicate_count = df_12.duplicated().sum()  # counts all duplicated rows
duplicate_df = df_12[df_12.duplicated(keep=False)].copy()  # makes df with dupe rows
df_12 = df_12.drop_duplicates() # drops row dupes
duplicate_pct = (len(duplicate_df) / len(df)) * 100  # calculates percentage of data that is dupelicated

print('\n', '-' * 50, '\n',
      ' ' * 16, 'Duplicate Rows:', duplicate_count, '\n',
      ' ' * 10, 'Duplicate Percentage:', duplicate_pct, '%\n',
      '-' * 50)





 -------------------------------------------------- 
                  Duplicate Rows: 0 
            Duplicate Percentage: 0.0 %
 --------------------------------------------------


In [4]:
# Inconsistent categories
print('\n', '-' * 50, '\n', ' ' * 15, 'Unique Check:\n', '-' * 50)
for col in df_12.select_dtypes(include='object').columns:  #find unique strings and their frequency
    print(f"\n{col}:")
    print(df_12[col].value_counts(dropna=False))
    print(df_12[col].unique())

# inconsistencies only found in CarName
df_12['make'] = df_12['CarName'].str.lower().str.strip().str.split().str[0] 
    #  makes lowercase     
    #  removes outer spaces
    #  extracts first word
df_12['make'] = df_12['make'].replace({'maxda': 'mazda', 'vokswagen': 'volkswagen', 'vw': 'volkswagen', 'porcshce': 'porsche'})  # add more as needed
    #  fixes spelling & inconcictency

print('\n', '-' * 50, '\n', ' ' * 15, 'Repared makes:\n', '-' * 50)
print(df_12['make'].value_counts(dropna=False))
print(df_12['make'].unique())


 -------------------------------------------------- 
                 Unique Check:
 --------------------------------------------------

CarName:
CarName
peugeot 504         5
toyota corolla      4
toyota corona       4
subaru dl           3
honda civic         3
                   ..
mazda glc 4         1
mazda rx2 coupe     1
maxda glc deluxe    1
maxda rx3           1
volvo 246           1
Name: count, Length: 127, dtype: int64
['alfa-romero giulia' 'alfa-romero stelvio' 'alfa-romero Quadrifoglio'
 'audi 100 ls' 'audi 100ls' 'audi fox' 'audi 5000' 'audi 4000'
 'audi 5000s (diesel)' 'bmw 320i' 'bmw x1' 'bmw z4' 'bmw x3'
 'chevrolet monte carlo' 'chevrolet vega 2300' 'dodge rampage'
 'dodge challenger se' 'dodge monaco (sw)' 'dodge colt hardtop'
 'dodge colt (sw)' 'dodge coronet custom' 'dodge dart custom'
 'dodge coronet custom (sw)' 'honda civic' 'honda civic cvcc'
 'honda accord cvcc' 'honda civic 1500 gl' 'honda accord'
 'honda civic 1300' 'honda prelude' 'honda civic (auto)' 'is

In [5]:
# unify data types
for col in df_12.columns:
    if col == 'CarName':  # skip car name
        continue
    if df_12[col].dtype == 'object':  # unify strings
        df_12[col] = df_12[col].astype(str).str.strip()
    elif pd.api.types.is_numeric_dtype(df_12[col]):  # unify numbers
        df_12[col] = pd.to_numeric(df_12[col], errors='coerce')

df_12.info()  # display uniform data types
df_12.describe(include='all')  # usefull info


<class 'pandas.core.frame.DataFrame'>
Index: 169 entries, 0 to 204
Data columns (total 27 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   car_ID            169 non-null    float64
 1   symboling         169 non-null    float64
 2   CarName           169 non-null    object 
 3   fueltype          169 non-null    object 
 4   aspiration        169 non-null    object 
 5   doornumber        169 non-null    object 
 6   carbody           169 non-null    object 
 7   drivewheel        169 non-null    object 
 8   enginelocation    169 non-null    object 
 9   wheelbase         169 non-null    float64
 10  carlength         169 non-null    float64
 11  carwidth          169 non-null    float64
 12  carheight         169 non-null    float64
 13  curbweight        169 non-null    float64
 14  enginetype        169 non-null    object 
 15  cylindernumber    169 non-null    object 
 16  enginesize        169 non-null    float64
 17  fu

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price,make
count,169.000000,169.000000,169,169,169,169,169,169,169,169.000000,...,169,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.00000,169.000000,169
unique,NaN,NaN,127,2,2,2,5,3,2,NaN,...,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21
top,NaN,NaN,peugeot 504,gas,std,four,sedan,fwd,front,NaN,...,mpfi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,toyota
freq,NaN,NaN,5,152,137,95,76,96,166,NaN,...,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23
mean,100.325444,0.881657,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98.665680,...,NaN,3.332308,3.256095,10.206568,104.715976,5146.449704,25.059172,30.56213,267370.907337,NaN
std,59.376277,1.233542,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.126456,...,NaN,0.273350,0.327050,4.054015,40.291081,476.737860,6.392615,6.81018,159581.131726,NaN
min,1.000000,-2.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86.600000,...,NaN,2.540000,2.070000,7.000000,52.000000,4150.000000,13.000000,16.00000,102360.000000,NaN
25%,50.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,94.500000,...,NaN,3.150000,3.150000,8.600000,70.000000,4800.000000,19.000000,25.00000,157900.000000,NaN
50%,100.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,96.900000,...,NaN,3.330000,3.270000,9.000000,95.000000,5200.000000,24.000000,30.00000,206900.000000,NaN
75%,150.000000,2.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101.200000,...,NaN,3.590000,3.410000,9.400000,116.000000,5500.000000,30.000000,34.00000,330300.000000,NaN


In [6]:
# Removing extreme values
Q1 = df_12['price'].quantile(0.25)
Q3 = df_12['price'].quantile(0.75)
IQR = Q3 - Q1  # inter quantile range
lower = Q1 - 1.5 * IQR  # lower limit
if lower < 0: lower = 0  # lower cant be below 0
upper = Q3 + 1.5 * IQR  # upper limit

extreme = df_12[(df_12['price'] < lower) | (df_12['price'] > upper)]

df_12 = df_12[(df_12['price'] >= lower) & (df_12['price'] <= upper)].copy()
    # remove extreme prices

print('\n', '-' * 50, '\n', 
      len(extreme), ' rows removed between ', lower, ' and ', upper,
      '\n', '-' * 50)



 -------------------------------------------------- 
 12  rows removed between  0  and  588900.0 
 --------------------------------------------------
